# Distributed (Masked) Dilithium Implementation

This notebook implements the **PQCDS 2** flow. Since the standard `liboqs` library does not support distributed/threshold signing out of the box (it is a black-box signer), we implement the **Dilithium logic flow** in Python to demonstrate the Host-Node interaction.

### Protocol Flow
1. **Setup:** Master Key $S_k$ is split into shares $s_{k_i}$ using Boolean Masking (XOR).
2. **Commit:** Nodes generate random $y_i$ and commit to $w_{1,i}$.
3. **Challenge:** Host aggregates $w_1$, decomposes it, and generates challenge $c$.
4. **Response:** Nodes compute partial responses $z_i$.
5. **Finalize:** Host aggregates $z$, computes Hint $h$, and forms Signature $\sigma$.

In [ ]:
import os
import hashlib
import secrets
from typing import List, Tuple

# --- Configuration ---
NUM_NODES = 3
Q = 8380417  # Dilithium Prime (Simplified for demonstration)
D = 13       # Dropped bits parameter (Simplified)

def shake256(data: bytes, length: int) -> bytes:
    return hashlib.shake_256(data).digest(length)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    """Utility to XOR two byte strings together."""
    return bytes(x ^ y for x, y in zip(a, b))

In [ ]:
class DilithiumLogic:
    """
    This class simulates the internal mathematical operations of Dilithium 
    required for the distributed flow (Commit, Decompose, Challenge, etc.).
    """
    
    @staticmethod
    def decompose(r: int):
        """Splits a value into High Bits (r1) and Low Bits (r0)."""
        # In real Dilithium: r = r1 * alpha + r0
        # We simulate this by bit splitting for the demo flow.
        alpha = 1 << D
        r0 = r % alpha
        r1 = (r - r0) // alpha
        if r0 > (alpha // 2):
            r0 -= alpha
            r1 += 1
        return r1, r0

    @staticmethod
    def pack_w1(w1_list: List[int]) -> bytes:
        """Encodes the High Bits (w1) into bytes."""
        b = bytearray()
        for w in w1_list:
            b.extend(w.to_bytes(4, 'big', signed=True))
        return bytes(b)

    @staticmethod
    def challenge(w1_high_bytes: bytes, message: bytes) -> bytes:
        """Generates the challenge c = H(w1_high || message)."""
        return shake256(w1_high_bytes + message, 32)

    @staticmethod
    def make_hint(z: int, r: int, c: int) -> int:
        """Constructs the hint h (simplified logic)."""
        # Real MakeHint checks if adding c causes an overflow in high bits
        # Here we return a simple indicator for the flow simulation
        return 1 if (z + c) > Q else 0

### 1. Setup Phase
The Host generates the Master Key and splits it into $N$ shares using **Boolean Masking**.
$s_{k_n} = (\bigoplus_{i=1}^{n-1} s_{k_i}) \oplus s_k$

In [ ]:
class Node:
    def __init__(self, node_id, sk_share, seed):
        self.id = node_id
        self.sk_share = sk_share  # This is sk_i
        self.seed = seed          # This is s_i
        self.y_current = None
    
    def round_1_commit(self, nonce_j: int) -> bytes:
        """
        Logic: 
        y_i = PRF(seed, j)
        w1_i = Commit(y_i) -> In this boolean masked flow, w1_i IS y_i
        """
        # Generate random vector y_i using PRF (SHAKE)
        # We simulate a vector of 4 integers for simplicity
        raw_y = shake256(self.seed + nonce_j.to_bytes(4, 'big'), 16)
        self.y_current = raw_y
        
        # In the masked flow, the node sends y_i (blinded/shared) as w1_i
        return self.y_current

    def round_2_response(self, challenge_c: bytes) -> bytes:
        """
        Logic:
        z_i = Response(y_i, c, sk_i)
        Using Masked Logic: z_i = y_i ^ (c * sk_i)
        """
        # Expand C and SK to match lengths for the XOR operation
        # Note: In real Dilithium, this is arithmetic (y + c*s). 
        # The PDF specifies XOR (Boolean Masking) for aggregation.
        
        # 1. Expand challenge to match vector size
        c_expanded = shake256(challenge_c, len(self.y_current))
        
        # 2. Simulate c * sk part (Masked Multiplication)
        # For this simulation, we combine c and sk_share cryptographically
        c_sk_mix = shake256(c_expanded + self.sk_share, len(self.y_current))
        
        # 3. Compute z_i = y_i XOR (c * sk)
        z_i = xor_bytes(self.y_current, c_sk_mix)
        return z_i

# --- Host Setup ---
message = b"Hello from Distributed Dilithium!"
master_sk = secrets.token_bytes(32)  # The real secret key
public_key = shake256(b"pk" + master_sk, 32)

nodes = []
accumulated_share = bytes(32)

# Split Secret Key into N shares
for i in range(NUM_NODES - 1):
    share = secrets.token_bytes(32)
    seed = secrets.token_bytes(32)
    nodes.append(Node(i, share, seed))
    accumulated_share = xor_bytes(accumulated_share, share)

# Last share calculation: sk_n = accumulated XOR master_sk
last_share = xor_bytes(accumulated_share, master_sk)
last_seed = secrets.token_bytes(32)
nodes.append(Node(NUM_NODES - 1, last_share, last_seed))

print(f"[Setup] Initialized {NUM_NODES} nodes with secret shares.")

### 2. Round 1: Commitment
Nodes calculate $y_i$ and send commitments. Host aggregates them: $w_1 = \bigoplus w_{1,i}$.

In [ ]:
nonce_j = 1
node_commitments = []

for node in nodes:
    w1_i = node.round_1_commit(nonce_j)
    node_commitments.append(w1_i)
    print(f"[Node {node.id}] Generated w1_i (len={len(w1_i)})")

# Host Aggregation: w1 = w1_0 XOR w1_1 XOR ...
w1_aggregated = bytes(len(node_commitments[0]))
for comm in node_commitments:
    w1_aggregated = xor_bytes(w1_aggregated, comm)

print(f"[Host] Aggregated w1: {w1_aggregated.hex()[:16]}...")

### 3. Round 2: Challenge
Host decomposes $w_1$ into HighBits ($w_1^H$), calculates Challenge $c = H(w_1^H || m)$, and broadcasts $c$.

In [ ]:
# 1. Decompose (Simulated)
# In the PDF flow, w1 is split. We convert bytes to integers to simulate 'HighBits'
w1_integers = [b for b in w1_aggregated]
w1_high = []
w1_low = []

for val in w1_integers:
    h, l = DilithiumLogic.decompose(val)
    w1_high.append(h)
    w1_low.append(l)

# 2. Compute Challenge c
w1_high_bytes = DilithiumLogic.pack_w1(w1_high)
challenge_c = DilithiumLogic.challenge(w1_high_bytes, message)

print(f"[Host] Computed Challenge c: {challenge_c.hex()[:16]}...")
print("[Host] Broadcasting c to all nodes...")

### 4. Round 3: Response
Nodes calculate $z_i$ and send it back. Host aggregates $z = \bigoplus z_i$ and generates Hint $h$.

In [ ]:
node_responses = []

for node in nodes:
    z_i = node.round_2_response(challenge_c)
    node_responses.append(z_i)
    print(f"[Node {node.id}] Computed response z_i")

# Host Aggregates z
z_aggregated = bytes(len(node_responses[0]))
for resp in node_responses:
    z_aggregated = xor_bytes(z_aggregated, resp)

print(f"[Host] Aggregated z: {z_aggregated.hex()[:16]}...")

# Host Creates Hint (Simplified)
# In real Dilithium, this checks for overflows between w1_low and z
# We generate a dummy hint for the structure
hint_h = shake256(b"hint" + z_aggregated, 16)

signature = {
    "z": z_aggregated,
    "c": challenge_c,
    "h": hint_h
}
print("\n=== Signature Generation Complete ===")
print(f"Signature: {signature}")

### 5. Verification
To verify, we check if the aggregated response $z$ and challenge $c$ mathematically align with the public key.

In this masked flow: $z = w_1 \oplus (c \cdot sk_{sum})$.
Therefore, $w_1$ should be recoverable as $w_1' = z \oplus (c \cdot pk)$ (conceptually).

In [ ]:
def verify_distributed_signature(pk, msg, sig):
    z = sig['z']
    c = sig['c']
    
    # 1. Recompute w1 (Inverse of the Node Response logic)
    c_expanded = shake256(c, len(z))
    # Note: In real system, PK is used here. 
    # Since we used a symmetric simulation (shared secret logic) for the demo:
    # We assume PK allows reconstructing the 'c * sk' term.
    # For this flow check, we simulate the reconstruction:
    c_pk_mix = shake256(c_expanded + master_sk, len(z))
    
    w1_recomputed_bytes = xor_bytes(z, c_pk_mix)
    
    # 2. Decompose Recomputed w1
    w1_int = [b for b in w1_recomputed_bytes]
    w1_high_re = []
    for val in w1_int:
        h, _ = DilithiumLogic.decompose(val)
        w1_high_re.append(h)
        
    # 3. Re-hash to check Challenge consistency
    w1_high_bytes = DilithiumLogic.pack_w1(w1_high_re)
    c_check = DilithiumLogic.challenge(w1_high_bytes, msg)
    
    print(f"Original c: {c.hex()[:16]}")
    print(f"Derived c:  {c_check.hex()[:16]}")
    
    return c == c_check

valid = verify_distributed_signature(public_key, message, signature)
print(f"\nVerification Result: {valid}")